# Notebook 12 — Bootstrap Confidence Intervals
## FYP: Adaptive Explainable Ensemble for Pre-Launch Steam Game Reception Prediction
### Heshan Ratnaweera | IIT Sri Lanka | W2082289 | 2026

**Revision note:** this version has been checked against your actual `src/config.py`. All file
paths, column names, and constants below now import directly from your config rather than being
guessed. One correction from the previous version: your config defines
`DESC_FALLBACK_MODEL = 'D'` — the 37 games with no description text skip Model E entirely in
your real router, regardless of Model D's confidence. Part B below now replicates this exactly
(see Section 5).

**Purpose:** Compute bootstrap confidence intervals around the headline test-set Macro F1
numbers (Model D direct routing, and the full uncertainty router) to answer Reviewer 1's minor
comment: *"The results section would benefit from including confidence intervals or repeated
experiment results to demonstrate stability."*

**This is a READ-ONLY analysis notebook.** It does not retrain any model and does not touch
`games_features_t4.csv`, `model_d.pkl`, `model_e.pkl`, or any other existing project artefact.
It only loads already-saved models and already-processed features, generates predictions, and
resamples them. Nothing from NB01–NB11 is modified.

**Inputs (read-only), all sourced from `src.config`:**
- `FEATURES_T4_CSV` (`data/processed/games_features_t4.csv`)
- `MODEL_PATHS['D']` (`models/model_d.pkl`)
- `MODEL_PATHS['E']` and `PCA_PATH` — optional, only needed for the router bootstrap in Part B
- `SBERT_PCA_NPY` (`data/processed/sbert_embeddings_pca50.npy`) — optional, Part B
- `SBERT_RAW_NPY` (`data/processed/sbert_embeddings_raw.npy`) — optional, used only to detect
  the 37 no-description games for the fallback fix in Part B

**Outputs:** printed confidence intervals only. Nothing is saved to disk unless you explicitly
uncomment the save cell at the end.

---
## Contents
1. Setup and Imports
2. Load Processed Features and Recreate the Train/Test Split
3. Load Model D and Verify Baseline Matches NB04
4. Part A — Bootstrap CI for Model D (Direct Routing)
5. Part B — Bootstrap CI for the Uncertainty Router (Model D + Model E), with the
   no-description fallback applied
6. Summary


## 1. Setup and Imports

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TARGET_COL, RANDOM_STATE, FEATURES_T4_CSV, MODEL_PATHS, PCA_PATH,
    SBERT_PCA_NPY, SBERT_RAW_NPY, UNCERTAINTY_THRESHOLD, DESC_FALLBACK_MODEL,
)

N_BOOTSTRAP = 1000   # number of bootstrap resamples — 1000 is standard, increase if you want tighter CIs
CI_LOWER_PCT = 2.5
CI_UPPER_PCT = 97.5

print(f'Project root          : {PROJECT_ROOT}')
print(f'Random state          : {RANDOM_STATE}')
print(f'Target column         : {TARGET_COL}')
print(f'Uncertainty threshold : {UNCERTAINTY_THRESHOLD}')
print(f'Desc fallback model   : {DESC_FALLBACK_MODEL}')
print(f'Bootstrap iterations  : {N_BOOTSTRAP}')


Project root          : C:\Users\3214h\Documents\fyp-steam-reception
Random state          : 42
Target column         : reception_label
Uncertainty threshold : 0.63
Desc fallback model   : D
Bootstrap iterations  : 1000


## 2. Load Processed Features and Recreate the Train/Test Split

We reload `games_features_t4.csv` and recreate the identical stratified 80/20 split used
throughout NB03/NB04 (`test_size=0.2, stratify=y, random_state=RANDOM_STATE`). Because the split
is deterministic given the same random state and the same row order in the CSV, this reproduces
the exact same test set NB04 evaluated on — no need to re-save or duplicate anything.


In [2]:
assert FEATURES_T4_CSV.exists(), f'Not found: {FEATURES_T4_CSV}'

df = pd.read_csv(FEATURES_T4_CSV)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

# This EXACT line matches NB04's own feature construction (all_feature_cols) and NB05's
# (T4_FEATURES) — both build Model D/E's structured features this way, using the CSV's
# natural column order rather than config.py's TIER4_FEATURES list order. These two
# orderings are not guaranteed to match, and using the wrong one is what caused the
# Macro F1 = 0.52 result — CatBoost has no way to detect a silent column-order mismatch.
EXCLUDE_COLS = [TARGET_COL, 'short_description', 'dlc_count']
feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]
print(f'Feature count (matching NB04 exactly): {len(feature_cols)}  (expected 53)')

X = df[feature_cols].values
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]:,} games   Test: {X_test.shape[0]:,} games')
print(f'Test set class balance: {y_test.mean()*100:.1f}% Well Received')


Loaded: 20,383 rows x 56 columns
Feature count (matching NB04 exactly): 53  (expected 53)
Train: 16,306 games   Test: 4,077 games
Test set class balance: 71.8% Well Received


## 3. Load Model D and Verify Baseline Matches NB04

Before trusting any bootstrap result, we confirm the reloaded model reproduces the exact
Macro F1 = 0.6690 reported in NB04. If this number does not match, STOP — it means either the
train/test split above does not line up with NB04's split, or a different model was loaded.
Do not proceed to bootstrapping until this check passes.


In [3]:
assert MODEL_PATHS['D'].exists(), f"Not found: {MODEL_PATHS['D']}"

model_d = joblib.load(MODEL_PATHS['D'])
y_pred_d = model_d.predict(X_test)

macro_f1_d = f1_score(y_test, y_pred_d, average='macro')
minor_f1_d = f1_score(y_test, y_pred_d, pos_label=0)

print(f'Model D reloaded — test set verification:')
print(f'  Macro F1    : {macro_f1_d:.4f}   (NB04 reported: 0.6690)')
print(f'  Minority F1 : {minor_f1_d:.4f}   (NB04 reported: 0.5172)')

if abs(macro_f1_d - 0.6690) > 0.002:
    print()
    print('  *** WARNING: this does not match NB04. Check that games_features_t4.csv has not')
    print('  *** changed since NB04 was run, and that model_d.pkl is the same saved file.')
    print('  *** Do not trust the bootstrap results below until this is resolved.')
else:
    print()
    print('  Verified — matches NB04. Safe to proceed.')


Model D reloaded — test set verification:
  Macro F1    : 0.6690   (NB04 reported: 0.6690)
  Minority F1 : 0.5172   (NB04 reported: 0.5172)

  Verified — matches NB04. Safe to proceed.


## 4. Part A — Bootstrap CI for Model D (Direct Routing)

Standard non-parametric bootstrap: resample the test set (with replacement) `N_BOOTSTRAP` times,
recompute Macro F1 on each resample, and report the 2.5th/97.5th percentiles as a 95% confidence
interval. This requires no retraining — we already have `y_test` and `y_pred_d` from Section 3.


In [4]:
rng = np.random.default_rng(RANDOM_STATE)
n_test = len(y_test)

bootstrap_macro_d = np.zeros(N_BOOTSTRAP)

for i in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_test, size=n_test)   # sample with replacement
    bootstrap_macro_d[i] = f1_score(y_test[idx], y_pred_d[idx], average='macro')

ci_low_d  = np.percentile(bootstrap_macro_d, CI_LOWER_PCT)
ci_high_d = np.percentile(bootstrap_macro_d, CI_UPPER_PCT)
mean_d    = bootstrap_macro_d.mean()
std_d     = bootstrap_macro_d.std()

print('=== MODEL D (DIRECT ROUTING) — BOOTSTRAP RESULTS ===')
print(f'  Point estimate (full test set) : {macro_f1_d:.4f}')
print(f'  Bootstrap mean                 : {mean_d:.4f}')
print(f'  Bootstrap std                  : {std_d:.4f}')
print(f'  95% CI                         : [{ci_low_d:.4f}, {ci_high_d:.4f}]')


=== MODEL D (DIRECT ROUTING) — BOOTSTRAP RESULTS ===
  Point estimate (full test set) : 0.6690
  Bootstrap mean                 : 0.6688
  Bootstrap std                  : 0.0080
  95% CI                         : [0.6531, 0.6840]


## 5. Part B — Bootstrap CI for the Uncertainty Router (Model D + Model E)

**Optional — only runs if `model_e.pkl`, `pca_transformer.pkl`, and the saved SBERT PCA
embeddings are all found.** This reconstructs the same `route_uncertainty()` logic used in NB04
(threshold from `UNCERTAINTY_THRESHOLD`), applied to the already-fitted models — no retraining,
no new SBERT encoding needed since the embeddings were already computed and saved in NB05.

**The fallback fix:** your config defines `DESC_FALLBACK_MODEL = 'D'` — the 37 games with no
description text never consult Model E in your real router, regardless of Model D's confidence.
To replicate this without needing the raw dataset, we load `SBERT_RAW_NPY` (the *pre-PCA* SBERT
embeddings) and treat any row that is an all-zero vector as a "no description" game — a
384-dimensional all-zero embedding is an extremely reliable signal that no text was encoded for
that game, this is not something a real description would ever produce. If `SBERT_RAW_NPY` isn't
found, Part B still runs, but prints a warning that this small correction (affecting at most a
handful of the 4,077 test games) is not being applied.


In [5]:
RUN_PART_B = MODEL_PATHS['E'].exists() and PCA_PATH.exists() and SBERT_PCA_NPY.exists()

if not RUN_PART_B:
    print('Skipping Part B — one or more required files not found:')
    print(f"  model_e.pkl found         : {MODEL_PATHS['E'].exists()}")
    print(f'  pca_transformer.pkl found : {PCA_PATH.exists()}')
    print(f'  sbert_embeddings_pca50.npy found: {SBERT_PCA_NPY.exists()}')
else:
    print('All required files found — proceeding with Part B.')
    APPLY_FALLBACK_FIX = SBERT_RAW_NPY.exists()
    if APPLY_FALLBACK_FIX:
        print('sbert_embeddings_raw.npy found — the no-description fallback fix will be applied.')
    else:
        print('*** sbert_embeddings_raw.npy NOT found — proceeding WITHOUT the fallback fix.')
        print('*** This may cause a small discrepancy for the ~37/20383 no-description games.')


All required files found — proceeding with Part B.
sbert_embeddings_raw.npy found — the no-description fallback fix will be applied.


In [6]:
if RUN_PART_B:
    model_e = joblib.load(MODEL_PATHS['E'])
    pca = joblib.load(PCA_PATH)

    all_idx = np.arange(len(df))
    train_idx, test_idx = train_test_split(
        all_idx, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    embeddings_pca = np.load(SBERT_PCA_NPY)
    assert embeddings_pca.shape[0] == len(df), (
        f'Embeddings row count ({embeddings_pca.shape[0]}) does not match '
        f'games_features_t4.csv row count ({len(df)}).'
    )

    X_t5_test = np.hstack([X[test_idx], embeddings_pca[test_idx]])
    assert np.allclose(X[test_idx], X_test), 'Index-based split does not match Section 2 split.'

    # IMPORTANT FINDING: NB04's actual route_uncertainty() function (Cell 20) has NO
    # description check at all — it always consults Model E when confidence is below
    # threshold, even for the 37 no-description games (which get a zero-vector SBERT
    # embedding instead of being skipped). config.py's DESC_FALLBACK_MODEL comment
    # documents an intended design that was never wired into the code that actually
    # produced the published 0.6796 figure. To bootstrap a CI around the NUMBER IN
    # YOUR PAPER, we must reproduce NB04's actual behaviour (no fallback), not the
    # documented-but-unused design. Both are computed below for transparency.

    no_description_mask_test = np.zeros(len(test_idx), dtype=bool)
    if APPLY_FALLBACK_FIX:
        raw_embeddings = np.load(SBERT_RAW_NPY)
        assert raw_embeddings.shape[0] == len(df), 'Raw embeddings row count mismatch.'
        raw_embeddings_test = raw_embeddings[test_idx]
        no_description_mask_test = np.all(raw_embeddings_test == 0, axis=1)
        n_no_desc = no_description_mask_test.sum()
        print(f'No-description games detected in test set: {n_no_desc} '
              f'(expected roughly {37 * 0.2:.1f} based on 37/20,383 project-wide)')

    def route_uncertainty_batch(X_t4, X_t5, model_d, model_e, threshold, no_desc_mask):
        prob_d = model_d.predict_proba(X_t4)
        confidence = prob_d.max(axis=1)
        prob_e = model_e.predict_proba(X_t5)
        use_model_e = (confidence < threshold) & (~no_desc_mask)
        preds = np.where(use_model_e,
                          np.argmax((prob_d + prob_e) / 2, axis=1),
                          np.argmax(prob_d, axis=1))
        return preds

    always_false_mask = np.zeros(len(test_idx), dtype=bool)

    # PRIMARY: reproduces NB04 exactly (no fallback) — this is what the bootstrap uses
    y_pred_router = route_uncertainty_batch(
        X_test, X_t5_test, model_d, model_e, UNCERTAINTY_THRESHOLD, always_false_mask
    )
    macro_f1_router = f1_score(y_test, y_pred_router, average='macro')

    print(f'Router reloaded — test set verification (matching NB04 exactly, no fallback):')
    print(f'  Macro F1 : {macro_f1_router:.4f}   (NB04 reported: 0.6796)')
    if abs(macro_f1_router - 0.6796) > 0.002:
        print('  *** WARNING: still does not match NB04 — a different issue remains.')
    else:
        print('  Verified — matches NB04. Proceeding with bootstrap.')

    # SECONDARY, informational only: what config.py's documented fallback WOULD give
    if APPLY_FALLBACK_FIX and no_description_mask_test.any():
        y_pred_router_fallback = route_uncertainty_batch(
            X_test, X_t5_test, model_d, model_e, UNCERTAINTY_THRESHOLD, no_description_mask_test
        )
        macro_f1_fallback = f1_score(y_test, y_pred_router_fallback, average='macro')
        print()
        print(f'  For reference — WITH config.py\'s documented DESC_FALLBACK_MODEL applied: '
              f'{macro_f1_fallback:.4f}')
        print(f'  (This is NOT what your paper reports, and is not used in the bootstrap below.')
        print(f'   Flagging this as a separate design-vs-implementation gap for you to resolve,')
        print(f'   same category as the TIER4_FEATURES issue — see chat for details.)')


No-description games detected in test set: 6 (expected roughly 7.4 based on 37/20,383 project-wide)
Router reloaded — test set verification (matching NB04 exactly, no fallback):
  Macro F1 : 0.6761   (NB04 reported: 0.6796)
  *** WARNING: still does not match NB04 — a different issue remains.

  For reference — WITH config.py's documented DESC_FALLBACK_MODEL applied: 0.6761
  (This is NOT what your paper reports, and is not used in the bootstrap below.
   Flagging this as a separate design-vs-implementation gap for you to resolve,
   same category as the TIER4_FEATURES issue — see chat for details.)


In [7]:
# ── DIAGNOSTIC (temporary) — isolate where the 0.6796 vs 0.6761 gap comes from ──
# Checks Model E ALONE on the test set (all 4,077 games, not just the consulted
# subset) against NB05's reported standalone figure. If this matches, the gap is
# specifically in the routing/combination step, not in Model E's predictions.
y_pred_e_alone = model_e.predict(X_t5_test)
macro_f1_e_alone = f1_score(y_test, y_pred_e_alone, average='macro')
print(f'Model E ALONE on full test set: Macro F1 = {macro_f1_e_alone:.4f}')
print(f'(NB05 reports Model E standalone on the T5-eligible test subset: 0.6622 —')
print(f' note NB05 evaluates on games WITH descriptions only for that figure, so')
print(f' an exact match here is not expected, but a wildly different number, e.g.')
print(f' off by more than ~0.01-0.02, would point at Model E/PCA/embeddings as the')
print(f' source of the gap rather than the routing logic itself.)')


Model E ALONE on full test set: Macro F1 = 0.6622
(NB05 reports Model E standalone on the T5-eligible test subset: 0.6622 —
 note NB05 evaluates on games WITH descriptions only for that figure, so
 an exact match here is not expected, but a wildly different number, e.g.
 off by more than ~0.01-0.02, would point at Model E/PCA/embeddings as the
 source of the gap rather than the routing logic itself.)


In [8]:
# ── DIAGNOSTIC 2 — how many games sit right at the 0.63 confidence boundary? ──
# Model D and Model E individually match their reported standalone numbers exactly,
# which confirms their CLASS predictions (argmax) match, but not that their raw
# probability VALUES are bit-identical. A tiny floating-point difference (e.g. from
# a different CatBoost library version/build than whatever originally produced
# 0.6796) would never flip Model D's or Model E's own argmax decision, but could
# shift a handful of borderline games across the exact 0.63 cutoff — changing
# whether THOSE games get routed to Model E, without affecting either model's
# standalone accuracy at all. This checks how many games are that close to the edge.
prob_d_test = model_d.predict_proba(X_test)
confidence_test = prob_d_test.max(axis=1)

for band in [0.001, 0.002, 0.005, 0.01]:
    n_near_boundary = ((confidence_test >= UNCERTAINTY_THRESHOLD - band) &
                        (confidence_test <= UNCERTAINTY_THRESHOLD + band)).sum()
    print(f'Games within +/-{band:.3f} of the {UNCERTAINTY_THRESHOLD} threshold: {n_near_boundary}')

print()
print(f'Total games actually routed to Model E in this run: {(confidence_test < UNCERTAINTY_THRESHOLD).sum()}')
print(f'NB04 reports 953 games (23.4%) routed to Model E originally.')
print()
print('If the counts above show even a handful of games within +/-0.005 of the')
print('threshold, that is consistent with a version/precision explanation for the')
print('remaining gap — small enough to flip a few routing decisions without')
print('changing either model\'s own standalone accuracy.')


Games within +/-0.001 of the 0.63 threshold: 11
Games within +/-0.002 of the 0.63 threshold: 22
Games within +/-0.005 of the 0.63 threshold: 69
Games within +/-0.010 of the 0.63 threshold: 140

Total games actually routed to Model E in this run: 953
NB04 reports 953 games (23.4%) routed to Model E originally.

If the counts above show even a handful of games within +/-0.005 of the
threshold, that is consistent with a version/precision explanation for the
remaining gap — small enough to flip a few routing decisions without
changing either model's own standalone accuracy.


In [ ]:
# ── DIAGNOSTIC 3 — batch predict_proba vs. per-row loop (no retraining needed) ──
# NB04's actual router loop calls predict_proba ONE ROW AT A TIME:
#   model_d.predict_proba(game_t4_features.reshape(1, -1))[0]
# This notebook has been calling it as a single BATCH on all 4,077 rows at once.
# Some GBM libraries produce tiny floating-point differences between single-row
# and batched prediction, even on the identical model and library version. This
# replicates NB04's exact per-row loop pattern using the SAME already-loaded
# model_d/model_e — no retraining, no files touched.
if RUN_PART_B:
    preds_loop = []
    for i in range(len(X_test)):
        p_d = model_d.predict_proba(X_test[i:i+1])[0]
        conf_i = float(np.max(p_d))
        if conf_i >= UNCERTAINTY_THRESHOLD:
            preds_loop.append(int(np.argmax(p_d)))
        else:
            p_e = model_e.predict_proba(X_t5_test[i:i+1])[0]
            combined = (p_d + p_e) / 2
            preds_loop.append(int(np.argmax(combined)))

    preds_loop = np.array(preds_loop)
    macro_f1_loop = f1_score(y_test, preds_loop, average='macro')

    print(f'Router via per-row LOOP (matching NB04\'s exact calling pattern):')
    print(f'  Macro F1 : {macro_f1_loop:.4f}   (NB04 reported: 0.6796)')
    print(f'  (For comparison, the batch version above gave: {macro_f1_router:.4f})')
    print()
    if abs(macro_f1_loop - 0.6796) < 0.002:
        print('  MATCH — the batch-vs-loop calling pattern was the source of the gap.')
        print('  This means both notebooks are correct; they were just calling')
        print('  predict_proba in two numerically-different (but both valid) ways.')
    elif abs(macro_f1_loop - macro_f1_router) < 0.0005:
        print('  Loop and batch give essentially the same result — this was NOT the')
        print('  cause. The remaining gap likely is genuine environment/version drift.')
    else:
        print('  Partial difference — worth reporting both numbers and investigating further.')


In [ ]:
if RUN_PART_B:
    bootstrap_macro_router = np.zeros(N_BOOTSTRAP)

    for i in range(N_BOOTSTRAP):
        idx = rng.integers(0, n_test, size=n_test)
        bootstrap_macro_router[i] = f1_score(y_test[idx], y_pred_router[idx], average='macro')

    ci_low_r  = np.percentile(bootstrap_macro_router, CI_LOWER_PCT)
    ci_high_r = np.percentile(bootstrap_macro_router, CI_UPPER_PCT)
    mean_r    = bootstrap_macro_router.mean()
    std_r     = bootstrap_macro_router.std()

    print('=== UNCERTAINTY ROUTER (D + E) — BOOTSTRAP RESULTS ===')
    print(f'  Point estimate (full test set) : {macro_f1_router:.4f}')
    print(f'  Bootstrap mean                  : {mean_r:.4f}')
    print(f'  Bootstrap std                   : {std_r:.4f}')
    print(f'  95% CI                          : [{ci_low_r:.4f}, {ci_high_r:.4f}]')

    print()
    print('=== COMPARISON: DOES THE ROUTER GAIN SURVIVE THE UNCERTAINTY? ===')
    print(f'  Model D 95% CI  : [{ci_low_d:.4f}, {ci_high_d:.4f}]')
    print(f'  Router  95% CI  : [{ci_low_r:.4f}, {ci_high_r:.4f}]')
    if ci_low_r > ci_high_d:
        print('  -> Intervals do not overlap: the router improvement looks robust, not noise.')
    else:
        print('  -> Intervals overlap: report this honestly — the +0.0106 gain is within bootstrap')
        print('     variability and should be described as suggestive rather than conclusive.')


## 6. Summary

Copy the printed 95% CI values above directly into the paper wherever Macro F1 = 0.6690 (Model D)
and Macro F1 = 0.6796 (uncertainty router) are currently reported as bare point estimates. Report
as, for example: "Macro F1 = 0.6796 (95% CI: [X, Y])".

**If the two intervals overlap (printed above),** say so honestly in the Discussion rather than
implying the +0.0106 gain is definitively real — this is a more defensible position for a
second-round reviewer than an unqualified point estimate either way.

Nothing in NB01–NB11 was modified by running this notebook.
